# MLP
在transformer中，在Attention之后，transformer做一个“自我思考”步骤：<br>
$x \rightarrow Linear \rightarrow \sigma \rightarrow 输出$ <br>
最早使用的是GELU: FFN(x) = W2(GELU(W1(x))) <br>
但是GELU的缺点是他把输入的所有维度混合在一起，没有门控能力。

## MLP的进化路线 <br>
1. GELU (GPT-2) <br>
2. GLU (Gated Linear Unit) <br>
3. GEGLU (GLU + GELU) <br>
4. SwiGLU (GLU +Silu)

## GLU：第一个革命点引入门控机制 <br>
GLU: $FFN(x) = W2(\sigma(W1(x)) \cdot W2(x))$ <br>
其中$\sigma(W1(x))$是门控特征<br>

## SwiGLU：第二个革命点引入SiLU激活函数 <br>
SiLU: $SiLU(x) = x \cdot \sigma(x)$ <br>
SwiGLU: $FFN(x) = W2(\sigma(W1(x)) \cdot SiLU(W2(x)))$ <br>

swiGLU:参数更少，训练更加稳定


In [2]:
import torch.nn as nn
from transformers.activations import ACT2FN


In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size

        self.gate_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.up_proj = nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
        self.down_proj = nn.Linear(self.intermediate_size, self.hidden_size, bias=False)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        gate = self.act_fn(self.gate_proj(x))
        x = self.up_proj(x)
        return self.down_proj(gate * x)